# PRISM - CONCH + CRC
# Pathology Reliability In Scarce-label Medicine

In [1]:
import torch
import numpy as np
import pandas as pd
import os
from google.colab import userdata, drive
from torchvision import transforms

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/PRISM'
EMBED_DIR = f'{BASE_DIR}/embeddings/conch/crc'
RESULTS_DIR = f'{BASE_DIR}/results'
os.makedirs(EMBED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.1 GB


In [2]:
!pip install transformers timm einops huggingface_hub scikit-learn scipy tqdm -q
!pip install git+https://github.com/Mahmoodlab/CONCH.git -q

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print("Login OK")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
Login OK


In [3]:
from conch.open_clip_custom import create_model_from_pretrained
model, transform = create_model_from_pretrained(
    'conch_ViT-B-16',
    checkpoint_path='hf_hub:MahmoodLab/CONCH',
    hf_auth_token=userdata.get('HF_TOKEN')
)
model = model.cuda().eval()
print("CONCH loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

CONCH loaded!
GPU memory: 1.58 GB


In [4]:
# CONCH provides its own transform above

In [6]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

CRC_DIR = '/content/drive/MyDrive/PRISM/datasets/crc'

# NCT-CRC-HE-100K → 70% train / 15% val / 15% test
full_dataset = ImageFolder(root=f'{CRC_DIR}/NCT-CRC-HE-100K', transform=transform)

train_size = int(0.70 * len(full_dataset))
val_size   = int(0.15 * len(full_dataset))
test_size  = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Classes: {full_dataset.classes}")
print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

Classes: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Train: 70,000
Val:   15,000
Test:  15,000


In [7]:
from tqdm import tqdm

def extract_features(model, loader, device='cuda'):
    all_features, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting"):
            images = images.to(device)
            features = model.encode_image(images, proj_contrast=False, normalize=True)
            features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy() if hasattr(labels, 'numpy') else np.array(labels))
    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting train features...")
train_features, train_labels = extract_features(model, train_loader)
print(f"Train: {train_features.shape}")

print("Extracting val features...")
val_features, val_labels = extract_features(model, val_loader)
print(f"Val: {val_features.shape}")

print("Extracting test features...")
test_features, test_labels = extract_features(model, test_loader)
print(f"Test: {test_features.shape}")

Extracting train features...


Extracting: 100%|██████████| 274/274 [2:06:08<00:00, 27.62s/it]


Train: (70000, 512)
Extracting val features...


Extracting: 100%|██████████| 59/59 [27:18<00:00, 27.77s/it]


Val: (15000, 512)
Extracting test features...


Extracting: 100%|██████████| 59/59 [27:44<00:00, 28.21s/it]

Test: (15000, 512)


In [8]:
np.save(f'{EMBED_DIR}/train_features.npy', train_features)
np.save(f'{EMBED_DIR}/train_labels.npy',   train_labels)
np.save(f'{EMBED_DIR}/val_features.npy',   val_features)
np.save(f'{EMBED_DIR}/val_labels.npy',     val_labels)
np.save(f'{EMBED_DIR}/test_features.npy',  test_features)
np.save(f'{EMBED_DIR}/test_labels.npy',    test_labels)
print(f"Embeddings saved to: {EMBED_DIR}")

Embeddings saved to: /content/drive/MyDrive/PRISM/embeddings/conch/crc


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

def compute_ece_multiclass(y_true, y_prob, n_bins=15):
    confidence  = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    correct     = (predictions == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidence >= bins[i]) & (confidence < bins[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correct[mask].mean() - confidence[mask].mean())
    return ece / len(y_true)

def run_label_fraction(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    y_prob    = clf.predict_proba(test_f)
    y_pred    = clf.predict(test_f)
    n_classes = y_prob.shape[1]
    return {
        'model': 'CONCH', 'dataset': 'CRC',
        'fraction': fraction, 'n_samples': n, 'seed': seed,
        'auroc': roc_auc_score(test_l, y_prob, multi_class='ovr', average='macro'),
        'f1':    f1_score(test_l, y_pred, average='macro'),
        'ece':   compute_ece_multiclass(test_l, y_prob),
        'brier': np.mean([brier_score_loss((test_l==c).astype(int), y_prob[:,c]) for c in range(n_classes)]),
    }

fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
seeds     = [42, 123, 456]
results   = []

for frac in fractions:
    for seed in seeds:
        res = run_label_fraction(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        results.append(res)
        print(f"  {frac*100:.0f}% | seed {seed} | AUROC: {res['auroc']:.4f} | ECE: {res['ece']:.4f} | Brier: {res['brier']:.4f}")

df = pd.DataFrame(results)
print("\n--- Summary (mean over seeds) ---")
print(df.groupby('fraction')[['auroc','ece','brier','f1']].mean().round(4))

  1% | seed 42 | AUROC: 0.9981 | ECE: 0.1877 | Brier: 0.0120
  1% | seed 123 | AUROC: 0.9981 | ECE: 0.1904 | Brier: 0.0118
  1% | seed 456 | AUROC: 0.9981 | ECE: 0.1868 | Brier: 0.0117
  5% | seed 42 | AUROC: 0.9987 | ECE: 0.0692 | Brier: 0.0060
  5% | seed 123 | AUROC: 0.9988 | ECE: 0.0695 | Brier: 0.0059
  5% | seed 456 | AUROC: 0.9986 | ECE: 0.0683 | Brier: 0.0059
  10% | seed 42 | AUROC: 0.9989 | ECE: 0.0471 | Brier: 0.0050
  10% | seed 123 | AUROC: 0.9989 | ECE: 0.0445 | Brier: 0.0049
  10% | seed 456 | AUROC: 0.9989 | ECE: 0.0434 | Brier: 0.0048
  25% | seed 42 | AUROC: 0.9992 | ECE: 0.0283 | Brier: 0.0040
  25% | seed 123 | AUROC: 0.9992 | ECE: 0.0273 | Brier: 0.0040
  25% | seed 456 | AUROC: 0.9991 | ECE: 0.0266 | Brier: 0.0040
  50% | seed 42 | AUROC: 0.9993 | ECE: 0.0192 | Brier: 0.0035
  50% | seed 123 | AUROC: 0.9993 | ECE: 0.0195 | Brier: 0.0035
  50% | seed 456 | AUROC: 0.9993 | ECE: 0.0190 | Brier: 0.0035
  100% | seed 42 | AUROC: 0.9994 | ECE: 0.0133 | Brier: 0.0031
  1

In [10]:
from scipy.optimize import minimize_scalar

def find_temperature_multiclass(val_logits, val_labels):
    def nll(T):
        scaled = val_logits / T
        exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
        probs  = exp_s / exp_s.sum(axis=1, keepdims=True)
        return -np.log(probs[np.arange(len(val_labels)), val_labels] + 1e-10).mean()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def run_temperature_scaling(train_f, train_l, val_f, val_l, test_f, test_l, fraction, seed=42):
    np.random.seed(seed)
    n   = int(len(train_l) * fraction)
    idx = np.random.choice(len(train_l), n, replace=False)
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    clf.fit(train_f[idx], train_l[idx])
    val_logits    = clf.decision_function(val_f)
    T             = find_temperature_multiclass(val_logits, val_l)
    test_prob_raw = clf.predict_proba(test_f)
    ece_raw       = compute_ece_multiclass(test_l, test_prob_raw)
    test_logits   = clf.decision_function(test_f)
    scaled        = test_logits / T
    exp_s         = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    test_prob_s   = exp_s / exp_s.sum(axis=1, keepdims=True)
    ece_scaled    = compute_ece_multiclass(test_l, test_prob_s)
    n_classes     = test_prob_raw.shape[1]
    return {
        'model': 'CONCH', 'dataset': 'CRC',
        'fraction': fraction, 'seed': seed,
        'temperature': T,
        'ece_raw': ece_raw, 'ece_scaled': ece_scaled,
        'ece_improvement': ece_raw - ece_scaled,
        'auroc':       roc_auc_score(test_l, test_prob_raw, multi_class='ovr', average='macro'),
        'brier_raw':   np.mean([brier_score_loss((test_l==c).astype(int), test_prob_raw[:,c]) for c in range(n_classes)]),
        'brier_scaled':np.mean([brier_score_loss((test_l==c).astype(int), test_prob_s[:,c])   for c in range(n_classes)]),
    }

ts_results = []
for frac in fractions:
    for seed in seeds:
        res = run_temperature_scaling(
            train_features, train_labels,
            val_features, val_labels,
            test_features, test_labels,
            fraction=frac, seed=seed
        )
        ts_results.append(res)

df_ts = pd.DataFrame(ts_results)
print("--- Temperature Scaling Results ---")
print(df_ts.groupby('fraction')[['temperature','ece_raw','ece_scaled','ece_improvement']].mean().round(4))

--- Temperature Scaling Results ---
          temperature  ece_raw  ece_scaled  ece_improvement
fraction                                                   
0.01           0.4328   0.1883      0.0039           0.1844
0.05           0.5961   0.0690      0.0039           0.0652
0.10           0.6553   0.0450      0.0037           0.0413
0.25           0.7163   0.0274      0.0029           0.0245
0.50           0.7489   0.0192      0.0024           0.0168
1.00           0.7823   0.0133      0.0019           0.0114


In [11]:
df.to_csv(f'{RESULTS_DIR}/conch_crc_results.csv', index=False)
df_ts.to_csv(f'{RESULTS_DIR}/conch_crc_temperature_scaling.csv', index=False)
print("Results saved!")
print(f"  {RESULTS_DIR}/conch_crc_results.csv")
print(f"  {RESULTS_DIR}/conch_crc_temperature_scaling.csv")

Results saved!
  /content/drive/MyDrive/PRISM/results/conch_crc_results.csv
  /content/drive/MyDrive/PRISM/results/conch_crc_temperature_scaling.csv
